# Lab 5 — Machine Learning for Intelligent Decision Making

**Student:** Hitesh Gupta | **Roll No.:** IC-2k23-39

Decision Tree Classifier on the Iris dataset (and optionally the
placement dataset). Generates confusion matrix, classification report,
and decision tree visualisations. Evaluates hyperparameter tuning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score,
                             f1_score)
from sklearn.preprocessing import LabelEncoder
import pickle, os

sns.set_style('whitegrid')
DATA_DIR = '../datasets'
OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Load and Prepare Data

In [ ]:
iris = pd.read_csv(f'{DATA_DIR}/iris.csv')
print('Shape:', iris.shape)
print(iris.head())
print('\nClass distribution:')
print(iris.iloc[:, -1].value_counts())

In [ ]:
# Separate features and target
feature_cols = [c for c in iris.columns if c != iris.columns[-1]]
X = iris[feature_cols].values
y_raw = iris[iris.columns[-1]].values
le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. Train Decision Tree (default parameters)

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred, average='weighted'), 4))
print('Recall   :', round(recall_score(y_test, y_pred, average='weighted'), 4))
print('F1-score :', round(f1_score(y_test, y_pred, average='weighted'), 4))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=class_names))

## 3. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Decision Tree (Iris)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Decision Tree Visualisation

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(dt, feature_names=feature_cols, class_names=class_names,
          filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree — Iris Classification')
plt.savefig(f'{OUT_DIR}/decision_tree.png', dpi=300, bbox_inches='tight')
plt.show()

print(export_text(dt, feature_names=feature_cols, max_depth=3))

## 5. Hyperparameter Tuning (max_depth, criterion)

In [ ]:
results = []
for criterion in ['gini', 'entropy']:
    for depth in [None, 2, 3, 4, 5, 6, 8, 10]:
        clf = DecisionTreeClassifier(criterion=criterion,
                                     max_depth=depth, random_state=42)
        scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
        results.append({
            'criterion': criterion,
            'max_depth': str(depth),
            'mean_acc': scores.mean(),
            'std_acc': scores.std(),
        })

tune_df = pd.DataFrame(results)
print(tune_df.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, crit in zip(axes, ['gini', 'entropy']):
    sub = tune_df[tune_df['criterion'] == crit]
    depths = [None if d == 'None' else int(d) for d in sub['max_depth']]
    ax.errorbar(range(len(sub)), sub['mean_acc'], yerr=sub['std_acc'],
                marker='o', capsize=4)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub['max_depth'])
    ax.set_xlabel('max_depth')
    ax.set_ylabel('CV Accuracy')
    ax.set_title(f'criterion={crit}')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/hyperparameter_tuning.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Save Model

In [ ]:
# Retrain best model and save
best = tune_df.loc[tune_df['mean_acc'].idxmax()]
best_depth = None if best['max_depth'] == 'None' else int(best['max_depth'])
best_crit = best['criterion']
print('Best: criterion=%s, max_depth=%s, acc=%.4f' % (best_crit, best_depth, best['mean_acc']))

final_model = DecisionTreeClassifier(criterion=best_crit,
                                     max_depth=best_depth, random_state=42)
final_model.fit(X, y)

model_path = '../models/decision_tree.pkl'
os.makedirs(os.path.dirname(model_path), exist_ok=True)
with open(model_path, 'wb') as f:
    pickle.dump(final_model, f)
print(f'Model saved to {model_path}')

## Summary

- Decision Tree achieves ~95–100% accuracy on Iris with default params.
- `petal_length` and `petal_width` are the most discriminative features.
- Limiting `max_depth` reduces overfitting; `entropy` and `gini` give
  similar results on this dataset.
- Cross-validation provides a more reliable estimate than a single split.